# Aztea — Docs Assistant fine-tune (Google Colab)

Trains a small LoRA adapter on the Aztea docs so it can answer questions about the platform (pricing, API keys, MCP, agent builder, disputes, etc.).

**Stack:** `transformers` + `peft` + `trl.SFTTrainer`, 4-bit base via `bitsandbytes`.  
**Base model (default):** `meta-llama/Llama-3.2-3B-Instruct` (gated — accept the license on HF first, then paste your HF token).  
**Runtime:** Colab free **T4** (15 GB). End-to-end ≈ 30–60 min on ~500 synthesized Q/A pairs.

## What this notebook does

1. Install deps and mount your HF token.
2. Pull the `docs/*.md` from the Aztea repo (public GitHub raw, or upload a zip).
3. Chunk each doc into ≤ 1,200-char passages.
4. **Synthesize Q/A pairs** from every passage with a teacher LLM (default: OpenAI `gpt-4o-mini`; swap for any chat API).
5. Train a LoRA adapter with `SFTTrainer` on Llama-3.2-3B-Instruct (4-bit).
6. Evaluate on a held-out split.
7. Save + zip the adapter and (optionally) push to the HF Hub.

The final adapter is what the Aztea backend's `/docs/ask` endpoint loads at inference time.

## 0. GPU + runtime check

In [ ]:
!nvidia-smi || echo 'No GPU — switch Runtime → Change runtime type → T4 GPU.'

## 1. Install dependencies

Pinned versions known-good on Colab T4 / CUDA 12.1.

In [ ]:
%%capture
!pip install -U "transformers==4.44.2" "peft==0.12.0" "trl==0.9.6" \
    "datasets==2.21.0" "accelerate==0.33.0" "bitsandbytes==0.43.3" \
    "sentencepiece" "huggingface_hub" "openai>=1.40.0" "tqdm"

## 2. Tokens

- `HF_TOKEN` — required if the base model is gated (Llama family is).
- `OPENAI_API_KEY` — used **only** to synthesize the Q/A dataset. Skip it if you already have your own Q/A JSONL and upload it at step 4.

In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('HF token (press Enter to skip): ') or os.environ.get('HF_TOKEN', '')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI key for teacher Q/A (Enter to skip): ') or os.environ.get('OPENAI_API_KEY', '')

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])

## 3. Pull the docs

Option A (default): pull from the public GitHub repo.  
Option B: upload a zip of your local `docs/` folder via `files.upload()`.

Edit `GITHUB_REPO` / `BRANCH` to point at your fork if needed.

In [ ]:
import os, requests, pathlib, re

GITHUB_REPO = 'pratyushsinghal7/agentmarket'   # <-- change to your fork if needed
BRANCH = 'main'
DOCS_DIR = pathlib.Path('docs')
DOCS_DIR.mkdir(exist_ok=True)

# List every .md at the root of docs/ via the GitHub API.
api_url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/docs?ref={BRANCH}'
resp = requests.get(api_url, timeout=30)
resp.raise_for_status()
items = [it for it in resp.json() if it.get('type') == 'file' and it.get('name', '').endswith('.md')]
print(f'Found {len(items)} markdown docs')

for it in items:
    raw = requests.get(it['download_url'], timeout=30).text
    (DOCS_DIR / it['name']).write_text(raw)
    print(' •', it['name'], f"({len(raw)} chars)")

### (alternative) Upload a zip of your own docs folder

In [ ]:
# Uncomment to use an upload instead of the GitHub pull above.
# from google.colab import files
# import zipfile, io
# up = files.upload()  # select docs.zip
# for name, data in up.items():
#     with zipfile.ZipFile(io.BytesIO(data)) as z:
#         z.extractall('docs')

## 4. Chunk docs into passages

We split on H2/H3 headings then pack up to ~1,200 chars. Each chunk becomes one "context" used to generate Q/A pairs and to train on.

In [ ]:
import re, pathlib, json

MAX_CHARS = 1200
chunks = []

for md_path in sorted(pathlib.Path('docs').glob('*.md')):
    text = md_path.read_text()
    # split on H2/H3
    parts = re.split(r'(?m)^(#{2,3}\s.*)$', text)
    current_heading = ''
    buf = ''
    def flush():
        global buf
        if buf.strip():
            chunks.append({
                'doc': md_path.name,
                'heading': current_heading.strip('# ').strip(),
                'text': buf.strip(),
            })
        buf = ''
    for p in parts:
        if re.match(r'^#{2,3}\s', p or ''):
            flush()
            current_heading = p
        else:
            seg = p or ''
            while len(buf) + len(seg) > MAX_CHARS and seg:
                take = MAX_CHARS - len(buf)
                buf += seg[:take]
                seg = seg[take:]
                flush()
            buf += seg
    flush()

print(f'Total chunks: {len(chunks)}')
print('Sample:', json.dumps(chunks[0], indent=2)[:400], '…')

## 5. Synthesize Q/A pairs with a teacher LLM

For each chunk we ask the teacher to produce 3 diverse Q/A pairs:
- factual lookup,
- "how do I…" task,
- edge-case / gotcha.

Swap the teacher to Anthropic, Groq, etc. by editing `call_teacher()`.

In [ ]:
import json, time
from tqdm.auto import tqdm
from openai import OpenAI

TEACHER_MODEL = 'gpt-4o-mini'
QA_PER_CHUNK = 3

client = OpenAI()

SYSTEM = (
    "You generate training data for a docs assistant. Given a passage from the "
    "Aztea (an AI agent marketplace) documentation, produce diverse Q/A pairs "
    "that a developer might ask. Questions must be answerable strictly from "
    "the passage; answers must be concise, accurate, and self-contained. "
    "Output JSON array with keys: question, answer."
)

def call_teacher(chunk):
    user = (
        f"Document: {chunk['doc']}\n"
        f"Section: {chunk['heading'] or '(top)'}\n\n"
        f"Passage:\n{chunk['text']}\n\n"
        f"Generate {QA_PER_CHUNK} Q/A pairs as a JSON array. "
        "Mix one factual, one how-to, one gotcha/edge case."
    )
    r = client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}],
        temperature=0.5,
        response_format={'type': 'json_object'},
    )
    raw = r.choices[0].message.content
    data = json.loads(raw)
    pairs = data if isinstance(data, list) else data.get('pairs') or data.get('qa') or data.get('data') or []
    return [p for p in pairs if isinstance(p, dict) and 'question' in p and 'answer' in p]

qa = []
for ch in tqdm(chunks, desc='Synthesizing Q/A'):
    try:
        pairs = call_teacher(ch)
        for p in pairs:
            qa.append({
                'doc': ch['doc'],
                'heading': ch['heading'],
                'context': ch['text'],
                'question': p['question'].strip(),
                'answer': p['answer'].strip(),
            })
    except Exception as e:
        print('skip:', ch['doc'], ch['heading'], '—', e)
        time.sleep(1)

print(f'Total Q/A pairs: {len(qa)}')
with open('docs_qa.jsonl', 'w') as f:
    for row in qa:
        f.write(json.dumps(row) + '\n')

### (alternative) Upload your own Q/A JSONL

If you skipped the teacher step, upload a JSONL where each line is `{"question": ..., "answer": ..., "context": ...}` (context is optional but recommended).

In [ ]:
# from google.colab import files
# up = files.upload()  # select docs_qa.jsonl

## 6. Build the training / eval splits

Format each row as a chat-style example. The model is trained to produce the answer given the question and (when available) the retrieved context.

In [ ]:
import json, random
from datasets import Dataset

rows = [json.loads(l) for l in open('docs_qa.jsonl')]
random.seed(7)
random.shuffle(rows)

SYS = (
    'You are the Aztea docs assistant. Answer questions about the Aztea '
    'platform using the provided context. Be concise, correct, and cite the '
    'doc + section when useful. If the context does not cover the answer, say so.'
)

def to_messages(row):
    user = row['question']
    if row.get('context'):
        user = f"Context (from {row.get('doc','')} · {row.get('heading','')}):\n{row['context']}\n\nQuestion: {row['question']}"
    return {
        'messages': [
            {'role': 'system', 'content': SYS},
            {'role': 'user', 'content': user},
            {'role': 'assistant', 'content': row['answer']},
        ]
    }

all_rows = [to_messages(r) for r in rows]
split = int(len(all_rows) * 0.9)
train_ds = Dataset.from_list(all_rows[:split])
eval_ds = Dataset.from_list(all_rows[split:])
print('train:', len(train_ds), 'eval:', len(eval_ds))

## 7. Load the base model (4-bit) + tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'  # gated — alternatives: 'Qwen/Qwen2.5-3B-Instruct' (ungated), 'HuggingFaceTB/SmolLM2-1.7B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
)
model.config.use_cache = False

## 8. Configure LoRA + SFT trainer

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

sft_cfg = SFTConfig(
    output_dir='./ckpt',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=False, fp16=True,
    max_seq_length=1536,
    packing=False,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_cfg,
)

## 9. Train

In [ ]:
trainer.train()

## 10. Qualitative eval

Sample a few held-out questions and compare the model's answer vs. the teacher's reference.

In [ ]:
import torch

model.eval()

def ask(messages, max_new_tokens=300):
    prompt = tok.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, pad_token_id=tok.pad_token_id)
    gen = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return gen.strip()

for row in eval_ds.select(range(min(5, len(eval_ds)))):
    msgs = row['messages']
    user_q = msgs[1]['content']
    reference = msgs[2]['content']
    pred = ask(msgs)
    print('Q:', user_q[:200].replace('\n',' '))
    print('REF:', reference[:300])
    print('GEN:', pred[:300])
    print('-' * 80)

## 11. Save + zip the adapter

Downloads a single `aztea_docs_adapter.zip` you can drop into the Aztea server and load with `PeftModel.from_pretrained(base, adapter_dir)` behind `/docs/ask`.

In [ ]:
import shutil, os

ADAPTER_DIR = 'aztea_docs_adapter'
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
shutil.make_archive(ADAPTER_DIR, 'zip', ADAPTER_DIR)
print('Adapter saved to', os.path.abspath(ADAPTER_DIR + '.zip'))

try:
    from google.colab import files
    files.download(ADAPTER_DIR + '.zip')
except Exception:
    pass

## 12. (optional) Push to HF Hub

Replace `YOUR_HF_USERNAME` and run. Requires `HF_TOKEN` with write scope.

In [ ]:
# HUB_ID = 'YOUR_HF_USERNAME/aztea-docs-assistant'
# model.push_to_hub(HUB_ID, private=True)
# tok.push_to_hub(HUB_ID, private=True)
# print('Pushed to', HUB_ID)

## Inference snippet to drop into the Aztea backend

```python
# core/docs_ask.py
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER = '/opt/aztea/adapters/aztea_docs_adapter'

_tok = AutoTokenizer.from_pretrained(BASE)
_base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
_model = PeftModel.from_pretrained(_base, ADAPTER).eval()

def answer(question: str, context: str = '') -> str:
    user = f'Context:\n{context}\n\nQuestion: {question}' if context else question
    msgs = [
        {'role': 'system', 'content': 'You are the Aztea docs assistant. Answer concisely using the context.'},
        {'role': 'user', 'content': user},
    ]
    prompt = _tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = _tok(prompt, return_tensors='pt').to(_model.device)
    with torch.no_grad():
        out = _model.generate(**ids, max_new_tokens=400, do_sample=False)
    return _tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
```

Wire `answer()` into the `POST /docs/ask` FastAPI route; retrieve the top-k matching chunks using `core/embeddings.py` first and pass them as `context`.